# 免矩阵求解特征值方法

## 算法实现

In [2]:

using Arpack, LinearOperators, LinearAlgebra

#定义算子乘法

function av!(res, A, v)
    res .= A*v
    return res
end

const N = 20
A = rand(N, N)
#op = LinearOperator(type, nrows, ncols, symmetric, hermitian, prod, tprod, ctprod)
# symmetric对应是否对称 hermitian对应是否Hermitian prod对应Av, tprod对应A^Tv, ctprod对应A^Hv
opA = LinearOperator(Float64, N, N, false, false, (res, v)->av!(res, A, v))
v = ones(N)
@show opA*v - A*v #opA这里相当于是一个matrix-free的算子 av!对应的是乘法res=Av

d1, = eigs(A);
d2, = eigs(opA);

@show [d1 d2]

opA * v - A * v = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[d1 d2] = 

ComplexF64[10.207526783161365 + 0.0im 10.207526783161368 + 0.0im; 1.1453478211233463 + 0.7363042679670557im 1.1453478211233485 + 0.7363042679670558im; 1.1453478211233463 - 0.7363042679670557im 1.1453478211233485 - 0.7363042679670558im; -1.1231338616425404 + 0.0im -1.1231338616425492 + 0.0im; 0.3654806407464859 + 1.0220574034072054im 0.3654806407464844 + 1.0220574034072047im; 0.3654806407464859 - 1.0220574034072054im 0.3654806407464844 - 1.0220574034072047im]


6×2 Matrix{ComplexF64}:
  10.2075+0.0im        10.2075+0.0im
  1.14535+0.736304im   1.14535+0.736304im
  1.14535-0.736304im   1.14535-0.736304im
 -1.12313+0.0im       -1.12313+0.0im
 0.365481+1.02206im   0.365481+1.02206im
 0.365481-1.02206im   0.365481-1.02206im

由此可见，使用LinearOperators模块定义的matrix-free的算子矩阵可以直接被Arpack的eigs使用求解特征值问题

## 时间模式中的应用
对动力系统问题
$$
\frac{\partial f}{\partial t} + L f =0
$$

可以将其沿着时间离散，则可以写出如下关系式
$$
f^{n+1} = \tilde L f^n
$$
其中$f^n$和$f^{n+1}$的时间间隔为$\Delta t$

因此，隐式的写出$\tilde L$的算子函数，即从$f^n$到$f^{n+1}$的计算函数，并将其定义为矩阵算子，即可采用arpack来进行求解特征值。
如果其特征值为$\mu$，那么对应的时间模式特征值为$-\frac{\ln \mu}{\Delta t}$

证明：
对原问题如果设$f = \hat f {\rm e}^{-\lambda t}$
则原方程可写成 $L\hat f=\lambda \hat f$，其中$\lambda$为动力系统特征值（需要额外注意对流动稳定性通常其对应${\rm i}\omega t$）
而对于离散算子$\tilde L$，则可以给出如下关系：
$$
\tilde L f^{n} = f^{n+1} = f^n {\rm e}^{-\lambda \Delta t}
$$
显然，对特征值问题有$\mu = {\rm e}^{-\lambda \Delta t}$，即$\lambda = -\frac{\ln \mu}{\Delta t}$
